# Tratamento de Dados — Bank Customer Churn

**Objetivo deste notebook:** carregar o dataset bruto, investigar sua qualidade e aplicar as correções necessárias, deixando um dataset limpo salvo em `data/processed/` para ser usado nos próximos notebooks (análise exploratória de negócio e modelagem).

**Regra deste projeto:** este notebook só trata dados (tipos, ausentes, outliers, duplicatas). Cálculos de negócio, EDA orientada a insights e modelagem ficam em notebooks separados.

Dataset: [Bank Customer Churn Data](https://www.kaggle.com/datasets/pentakrishnakishore/bank-customer-churn-data) — 28.382 clientes de um banco.

In [48]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 1. Carregar os dados brutos

In [49]:
df = pd.read_csv(r"C:\Users\engpe\OneDrive\Documentos\Portfólio\bank-churn-prediction-dashboard\data\raw\churn_prediction.csv")
print(f"Formato: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head()

Formato: 28382 linhas x 21 colunas


,customer_id,vintage,age,gender,dependents,occupation,city,customer_nw_category,branch_code,current_balance,previous_month_end_balance,average_monthly_balance_prevQ,average_monthly_balance_prevQ2,current_month_credit,previous_month_credit,current_month_debit,previous_month_debit,current_month_balance,previous_month_balance,churn,last_transaction
0,1,2101,66,Male,0.0,self_employed,187.0,2,755,1458.71,1458.71,1458.71,1449.07,0.20,0.20,0.20,0.20,1458.71,1458.71,0,2019-05-21
1,2,2348,35,Male,0.0,self_employed,NaN,2,3214,5390.37,8704.66,7799.26,12419.41,0.56,0.56,5486.27,100.56,6496.78,8787.61,0,2019-11-01
2,4,2194,31,Male,0.0,salaried,146.0,2,41,3913.16,5815.29,4910.17,2815.94,0.61,0.61,6046.73,259.23,5006.28,5070.14,0,NaT
3,5,2329,90,NaN,NaN,self_employed,1020.0,2,582,2291.91,2291.91,2084.54,1006.54,0.47,0.47,0.47,2143.33,2291.91,1669.79,1,2019-08-06
4,6,1579,42,Male,2.0,self_employed,1494.0,3,388,927.72,1401.72,1643.31,1871.12,0.33,714.61,588.62,1538.06,1157.15,1677.16,1,2019-11-03


## 2. Visão geral: tipos de dado e estrutura

In [50]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28382 entries, 0 to 28381
Data columns (total 21 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   customer_id                     28382 non-null  int64  
 1   vintage                         28382 non-null  int64  
 2   age                             28382 non-null  int64  
 3   gender                          27857 non-null  str    
 4   dependents                      25919 non-null  float64
 5   occupation                      28302 non-null  str    
 6   city                            27579 non-null  float64
 7   customer_nw_category            28382 non-null  int64  
 8   branch_code                     28382 non-null  int64  
 9   current_balance                 28382 non-null  float64
 10  previous_month_end_balance      28382 non-null  float64
 11  average_monthly_balance_prevQ   28382 non-null  float64
 12  average_monthly_balance_prevQ2  28382 non-n

**Observação:** `last_transaction` veio como texto (`object`), mas é uma data — vamos converter na seção 5.

## 3. Duplicatas

In [51]:
print("Linhas duplicadas:", df.duplicated().sum())
print("customer_id duplicados:", df['customer_id'].duplicated().sum())

Linhas duplicadas: 0
customer_id duplicados: 0


**Decisão:** não há linhas nem `customer_id` duplicados. Nada a remover aqui.

## 4. Valores ausentes

In [52]:
ausentes = df.isnull().sum()
ausentes_pct = (ausentes / len(df) * 100).round(2)
pd.DataFrame({'ausentes': ausentes, 'pct': ausentes_pct}).query('ausentes > 0').sort_values('pct', ascending=False)

,ausentes,pct
dependents,2463,8.68
city,803,2.83
gender,525,1.85
occupation,80,0.28


**Decisões de tratamento** (cada uma documentada porque isso é parte do que vai para o README do projeto):

- **`dependents`** (8,68% ausente): número de dependentes. Vou preencher com a **mediana**, porque a distribuição tem outliers fortes (ver seção 6) que puxariam a média para um valor irreal.
- **`city`** (2,83% ausente): é um código numérico de cidade, não um nome — não existe uma "cidade mais comum" que faça sentido substituir. Vou marcar como uma categoria própria (`-1` = "não informado") em vez de inventar um valor.
- **`gender`** (1,85% ausente): vou preencher como `"Not Informed"` — mais honesto do que assumir o gênero mais frequente.
- **`occupation`** (0,28% ausente): proporção muito pequena. Vou preencher com a moda (`self_employed`), já que o impacto de errar aqui é mínimo.

In [53]:
df['dependents'] = df['dependents'].fillna(df['dependents'].median())
df['city'] = df['city'].fillna(-1)
df['gender'] = df['gender'].fillna('Not Informed')
df['occupation'] = df['occupation'].fillna(df['occupation'].mode()[0])

print("Ausentes restantes:")
print(df.isnull().sum().sum(), "células (esperado: 0, exceto o que tratarmos na seção 5)")

Ausentes restantes:
0 células (esperado: 0, exceto o que tratarmos na seção 5)


## 5. Corrigir tipos de dado

In [54]:
# last_transaction: converter para datetime de verdade.
# O valor "NaT" já vem como texto no CSV — o pandas vai reconhecer e
# transformar em data ausente de verdade (NaT do tipo datetime).
df['last_transaction'] = pd.to_datetime(df['last_transaction'], errors='coerce')

print("Clientes sem transação registrada:", df['last_transaction'].isnull().sum())
print("Data mais antiga:", df['last_transaction'].min())
print("Data mais recente:", df['last_transaction'].max())

Clientes sem transação registrada: 3223
Data mais antiga: 2018-12-31 00:00:00
Data mais recente: 2019-12-31 00:00:00


**Decisão:** não vou remover as linhas sem `last_transaction` — a ausência de transação recente pode, ela mesma, ser um sinal de risco de churn (vamos testar essa hipótese no notebook de EDA de negócio, não aqui).

In [55]:
# Colunas categóricas: converter para 'category' economiza memória e deixa
# explícito, pra quem lê o notebook, quais colunas são categóricas.
colunas_categoricas = ['gender', 'occupation', 'customer_nw_category']
for c in colunas_categoricas:
    df[c] = df[c].astype('category')

df[colunas_categoricas].dtypes

gender                  category
occupation              category
customer_nw_category    category
dtype: object

## 6. Outliers

In [56]:
df['dependents'].describe()

count    28382.000000
mean         0.317102
std          0.958386
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         52.000000
Name: dependents, dtype: float64

A mediana e o 75º percentil de `dependents` são baixos, mas o máximo chega a valores como 52 — claramente erro de digitação/cadastro, não uma família real. Vou tratar valores acima de 10 como inválidos, transformá-los em ausente e reaplicar a mediana.

In [57]:
outliers_dependents = (df['dependents'] > 10).sum()
print(f"Valores de 'dependents' acima de 10: {outliers_dependents}")

df.loc[df['dependents'] > 10, 'dependents'] = df['dependents'].median()

Valores de 'dependents' acima de 10: 5


In [58]:
print("Idade mínima:", df['age'].min(), "| Idade máxima:", df['age'].max())
print("Clientes com menos de 18 anos:", (df['age'] < 18).sum(), 
      f"({(df['age'] < 18).mean()*100:.1f}% da base)")

Idade mínima: 1 | Idade máxima: 90
Clientes com menos de 18 anos: 806 (2.8% da base)


**Decisão consciente, não automática:** há ~800 clientes (2,8%) com menos de 18 anos. Isso é alto demais para ser só erro de digitação — bancos na Índia (origem deste dataset) permitem contas de menores, geralmente geridas por um responsável. Por isso, **não vou remover nem alterar essas linhas**. Vou apenas documentar essa decisão para justificar no README do projeto, e deixar como um ponto de investigação para a análise de negócio (será que contas de menores têm padrão de churn diferente?).

In [59]:
saldo_negativo = (df['current_balance'] < 0).sum()
print(f"Clientes com saldo atual negativo: {saldo_negativo}")

# Em vez de remover ou "corrigir" o valor, transformo em uma feature própria.
# Saldo negativo pode ser um sinal de risco real (cheque especial, por exemplo),
# então apagar essa informação seria jogar fora um dado potencialmente útil.
df['saldo_negativo'] = (df['current_balance'] < 0).astype(int)
df['saldo_negativo'].value_counts()

Clientes com saldo atual negativo: 17


saldo_negativo
0    28365
1       17
Name: count, dtype: int64

## 7. Conferência final

In [60]:
print("Formato final:", df.shape)
print()
print("Ausentes restantes:")
print(df.isnull().sum())
print()
print("Taxa de churn:", f"{df['churn'].mean()*100:.2f}%")

Formato final: (28382, 22)

Ausentes restantes:
customer_id                          0
vintage                              0
age                                  0
gender                               0
dependents                           0
occupation                           0
city                                 0
customer_nw_category                 0
branch_code                          0
current_balance                      0
previous_month_end_balance           0
average_monthly_balance_prevQ        0
average_monthly_balance_prevQ2       0
current_month_credit                 0
previous_month_credit                0
current_month_debit                  0
previous_month_debit                 0
current_month_balance                0
previous_month_balance               0
churn                                0
last_transaction                  3223
saldo_negativo                       0
dtype: int64

Taxa de churn: 18.53%


## 8. Salvar dataset tratado

In [61]:
df.to_csv(r"C:\Users\engpe\OneDrive\Documentos\Portfólio\bank-churn-prediction-dashboard\data\processed\churn_clean.csv", index=False)
print("Salvo em data/processed/churn_clean.csv")

Salvo em data/processed/churn_clean.csv


---
### Resumo das decisões (para o README do projeto)

| Coluna | Problema | Decisão |
|---|---|---|
| `dependents` | 8,7% ausente + outliers (até 52) | outliers >10 e ausentes → mediana |
| `city` | 2,8% ausente | categoria própria (-1 = não informado) |
| `gender` | 1,9% ausente | categoria "Not Informed" |
| `occupation` | 0,3% ausente | moda (`self_employed`) |
| `last_transaction` | texto, com "NaT" | convertido para data real; ausentes mantidos (sinal potencial) |
| `age` | ~800 clientes < 18 anos | mantido — provável conta de menor, plausível no mercado indiano |
| `current_balance` | 17 saldos negativos | mantido + nova coluna `saldo_negativo` (flag) |

**Próximo notebook:** `02_eda_negocio.ipynb` — análise exploratória orientada às perguntas de negócio (perfil de quem cancela, cálculo de receita em risco).